# Phân loại EEG theo từng lần thử trong thí nghiệm oddball

## 1. Mục tiêu và dữ liệu

Thí nghiệm đánh giá khả năng phân biệt kích thích thường xuyên (*standard*) và kích thích mục tiêu (*target*) từ đáp ứng điện não đồ (EEG) của từng lần kích thích (trial). Bốn bộ phân loại được khảo sát gồm Shrinkage LDA, SVM RBF, Random Forest và Logistic Regression L2. Đại lượng so sánh chính là F1 của lớp target khi mô hình được kiểm tra trên những người không xuất hiện trong tập huấn luyện. Precision, recall, balanced accuracy, ROC-AUC và average precision (AP) được dùng để phân tích bổ sung.

Dữ liệu oddball thuộc bộ Nencki–Symfonia của [Dzianok và cộng sự (2022)](https://doi.org/10.1093/gigascience/giac015), gồm bản ghi của 42 người. Các file BrainVision và bảng sự kiện BIDS được lấy từ [duylplsn/oddball](https://huggingface.co/datasets/duylplsn/oddball). Phân tích này sử dụng hai trong ba điều kiện kích thích:

**Bảng 1.1. Quy tắc gán nhãn từ cột `event_type` trong BIDS.**

| Điều kiện | Mã sự kiện | Nhãn phân loại |
| :--- | :--- | :--- |
| Standard / frequent | S5 | 0 |
| Target | S6 | 1 |
| Distractor | S7 | Loại khỏi phân tích |

Một quan sát đầu vào là một epoch EEG; đơn vị phân chia tập huấn luyện/kiểm tra là **người tham gia**. Số người, số epoch và tỷ lệ lớp thực sự được sử dụng được xác định từ dữ liệu sau tiền xử lý ở Mục 3.2.

## 2. Môi trường tính toán

Mã nguồn được tổ chức thành các module Python; dữ liệu trung gian và kết quả được lưu trong `artifacts/` trên Google Drive. Bảng 2.1 ghi phiên bản mã nguồn, thư viện và trạng thái CUDA/cuML của lần chạy. Backend sử dụng thực tế được báo cáo riêng cho từng mô hình. Hướng dẫn cài đặt và khôi phục một lần chạy nằm trong [README](https://github.com/namkhanhdoan2305/Miniproject-Computational-NeuroScience#readme).

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/Miniproject-Computational-NeuroScience')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/namkhanhdoan2305/Miniproject-Computational-NeuroScience.git', str(REPO)], check=True)
else:
    changes = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True)
    if changes.strip():
        raise RuntimeError('Working tree có thay đổi chưa commit; dừng cập nhật mã nguồn.')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

ENABLE_GPU = True
has_nvidia = shutil.which('nvidia-smi') is not None
if ENABLE_GPU and has_nvidia:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-gpu.txt'], check=True)
    except subprocess.CalledProcessError:
        print('cuML chưa khả dụng; backend thực tế được ghi trong kết quả từng mô hình.')

DRIVE_OUTPUT = Path('/content/drive/MyDrive/oddball_artifacts')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
ART = REPO / 'artifacts'
if not ART.exists() and not ART.is_symlink():
    ART.symlink_to(DRIVE_OUTPUT, target_is_directory=True)
if not ART.is_symlink() or ART.resolve() != DRIVE_OUTPUT.resolve():
    raise RuntimeError('Thư mục artifacts không trỏ đến đường dẫn Drive đã cấu hình.')

from src.common import load_config, package_versions, write_json
from src.backends import gpu_status
from src.notebook import (run_step, show_model, show_qc, show_environment,
                          show_preprocessing_config, show_dataset, show_features,
                          show_protocol, show_comparison)
CONFIG = load_config()
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
environment = {'git_commit': revision, 'python': sys.version, 'versions': package_versions(), 'gpu': gpu_status()}
write_json(ART / 'environment.json', environment)
show_environment(ART)

## 3. Tiền xử lý và kiểm tra tín hiệu

### 3.1. Quy trình tiền xử lý

Các kênh EEG được chọn theo một danh sách cố định trước khi đánh giá mô hình. Kênh đã được BIDS đánh dấu hỏng được nội suy từ vị trí điện cực. Trình tự xử lý tín hiệu là:

**EEG liên tục → notch → bandpass → resample → ICA → đổi tham chiếu → epoch và baseline.**

ICA được ước lượng trên đoạn đầu bản ghi trong giới hạn thời lượng cấu hình, sử dụng một bản sao đã high-pass; sau đó áp dụng phép loại thành phần lên bản ghi dùng để tạo epoch. EOG được dùng làm tín hiệu tham chiếu để phát hiện thành phần liên quan đến mắt; khi thiếu EOG chuyên dụng, Fp1/Fp2 được dùng làm tín hiệu thay thế. Việc dùng high-pass riêng khi ước lượng ICA dựa trên [quy trình xử lý artifact của MNE](https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html). Tham số áp dụng được liệt kê trong Bảng 3.1.

Với $x_{ic}(t)$ là điện thế của trial $i$ tại kênh $c$, hiệu chỉnh baseline được tính theo:

$$
\widetilde{x}_{ic}(t)=x_{ic}(t)-\frac{1}{|\mathcal B|}\sum_{u\in\mathcal B}x_{ic}(u),
$$

trong đó $\mathcal B$ là các mẫu thuộc khoảng baseline trước kích thích. Epoch vượt ngưỡng peak-to-peak, gần phẳng hoặc cắt qua ranh giới bản ghi được loại. Tín hiệu được lưu theo volt; các hình biên độ được quy đổi sang µV.

Kiểm tra chi tiết được thực hiện trên sub-01. Các trace và PSD sử dụng cùng một đoạn ghi; ERP giữa các giai đoạn chỉ sử dụng các trial đồng thời có mặt trong tập được giữ lại. Trong các hình QC, `raw` chỉ giai đoạn sau chọn kênh/nội suy kênh BIDS nhưng trước các bộ lọc thời gian.

In [ ]:
show_preprocessing_config(CONFIG)

from src.common import read_json
meta_path = ART / 'metadata.json'
if not meta_path.exists() or read_json(meta_path)['dry_run']:
    run_step('src.preprocess', '--dry-run', '--resume')
elif not list((ART / 'qc' / 'sub-01').glob('*/metadata.json')):
    run_step('src.preprocess', '--qc-only', '--subjects', 'sub-01', '--revision', read_json(meta_path)['revision'])
show_qc(ART, 'sub-01')

### 3.2. Dữ liệu sau tiền xử lý và đáp ứng gợi

Mỗi người được xử lý độc lập; bản ghi thô được xóa sau khi lưu checkpoint. Bảng 3.3 mô tả tập epoch được đưa vào phân loại. Phân bố theo người cho biết mức lệch lớp và sự khác nhau về số trial còn lại.

Đáp ứng gợi (ERP) được tính bằng cách lấy trung bình trial trong từng người/lớp, rồi lấy trung bình giữa người với trọng số bằng nhau. Fz, Cz và Pz lần lượt là các vị trí đường giữa vùng trán, trung tâm và đỉnh đầu. Cửa sổ P300 được tô trên hình để đối chiếu biên độ và hình thái giữa hai lớp.

In [ ]:
from src.common import load_epochs
cohort_ready = False
if meta_path.exists():
    meta = read_json(meta_path)
    same_settings = (meta['config']['preprocessing'] == CONFIG['preprocessing']
                     and meta['config']['events'] == CONFIG['events']
                     and meta['config']['seed'] == CONFIG['seed'])
    cohort_ready = (not meta['dry_run'] and len(meta['subjects']) == CONFIG['dataset']['expected_subjects'] and same_settings)
if not cohort_ready:
    run_step('src.preprocess', '--resume')
load_epochs(ART)  # kiểm tra schema trước bước tiếp theo
run_step('src.visualize', '--data-only')
show_dataset(ART)

## 4. Đặc trưng và thiết kế đánh giá

### 4.1. Biểu diễn đặc trưng

Mỗi epoch được biểu diễn bởi các đặc trưng theo kênh và đặc trưng tổng hợp trên ROI Fz/Cz/Pz. Với cấu hình mặc định 30 kênh, 4 dải tần và wavelet 4 mức, vector đầu vào có $30\times26+5=785$ chiều. Bảng 4.1 thống kê số chiều thực tế của lần chạy.

| Nhóm | Cách tính | Đơn vị / khoảng tính |
| :--- | :--- | :--- |
| Công suất dải tần | Tích phân PSD Welch và lấy $\log_{10}$; delta, theta, alpha, beta | PSD tính từ tín hiệu µV; cửa sổ đặc trưng 0–1 s mặc định |
| Entropy | Entropy của PSD và histogram biên độ; chuẩn hóa theo số phần tử | Không đơn vị, khoảng 0–1 |
| Wavelet | Trung bình trị tuyệt đối, SD và log năng lượng của hệ số `db4` tại A4/D4/D3/D2/D1 | Cửa sổ đặc trưng; 15 thống kê/kênh khi dùng 4 mức |
| P300 theo kênh | Mean, peak dương, SD, peak latency, diện tích có dấu | µV, ms, µV·s; cửa sổ 300–600 ms mặc định |
| P300 ROI | Năm thống kê trên waveform trung bình Fz/Cz/Pz | Cùng cửa sổ và đơn vị với P300 theo kênh |

Diện tích P300 được tính bằng $A_{ic}=\int_{t_1}^{t_2}\widetilde{x}_{ic}(t)\,dt$, giữ nguyên dấu điện thế. Đại lượng này khác ROC-AUC của bộ phân loại. Peak và latency được xác định trong cửa sổ P300 đã chọn; các ước lượng trên từng trial có thể chịu ảnh hưởng của nhiễu nền.

In [ ]:
from src.features import load_features
if (ART / 'feature_metadata.json').exists():
    _, _, _, feature_names, feature_info = load_features(ART)
    if feature_info['settings'] != CONFIG['features']:
        raise RuntimeError('Cấu hình đặc trưng thay đổi. Chạy lại src.features chủ động trước khi train.')
else:
    run_step('src.features')
from src.visualize import plot_distributions
plot_distributions(ART, CONFIG['visualization'])
show_features(ART)

### 4.2. Phân chia theo người và chuẩn hóa

Đánh giá sử dụng [GroupKFold](https://scikit-learn.org/stable/modules/cross_validation.html#group-k-fold), mặc định $K=5$. Cùng một tập fold được dùng cho cả bốn mô hình. Với mỗi fold $k$:

$$
G_{\mathrm{train},k}\cap G_{\mathrm{test},k}=\varnothing,
\qquad
z_{ij}=\frac{\phi_j(x_i)-\mu_{j,\mathrm{train},k}}{\sigma_{j,\mathrm{train},k}}.
$$

$\phi_j(x_i)$ là đặc trưng thứ $j$ của trial $i$; trung bình và SD dùng để chuẩn hóa chỉ được ước lượng từ train fold. Mỗi trial được kiểm tra đúng một lần. Tham số mô hình được cố định theo Bảng 4.2; thí nghiệm hiện tại không tìm kiếm siêu tham số hoặc chọn ngưỡng trên test fold.

SVM, Random Forest và Logistic Regression sử dụng trọng số lớp $w_c=N_{\mathrm{train}}/(2N_{c,\mathrm{train}})$. LDA sử dụng xác suất tiên nghiệm được chỉ định trong cấu hình. Cách xử lý này tăng mức đóng góp của lớp target hiếm trong quá trình fit, nhưng không làm thay đổi tỷ lệ lớp của tập kiểm tra.

### 4.3. Chỉ số và đường chẩn đoán

Target là lớp dương. Hai chỉ số phản ánh trực tiếp việc nhận diện hai lớp là:

$$
F_1=\frac{2TP}{2TP+FP+FN},\qquad
\mathrm{BA}=\frac12\left(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\right).
$$

ROC-AUC và AP sử dụng điểm liên tục; precision, recall, F1 và confusion matrix sử dụng nhãn dự đoán tại ngưỡng mặc định của mô hình. Các chỉ số được tính riêng trên từng test fold và báo cáo dưới dạng trung bình ± SD mẫu ($ddof=1$). SD giữa các fold là đại lượng mô tả biến thiên, không phải khoảng tin cậy.

Đường học sử dụng một **inner split theo người** được lấy từ train của outer fold thứ nhất. Số người train được tăng theo các tập lồng nhau, còn inner validation được giữ cố định. LDA/SVM được khảo sát bằng learning curve; Random Forest được khảo sát thêm theo số cây. Với Logistic Regression, một lượt L-BFGS riêng trên CPU ghi lịch sử objective và log loss. Những đường này không sử dụng outer test và không tham gia lựa chọn tham số.

In [ ]:
show_protocol(CONFIG, ART)

## 5. Kết quả từng bộ phân loại

### 5.1. Shrinkage LDA

LDA xây dựng ranh giới tuyến tính từ trung bình từng lớp và ma trận hiệp phương sai chung. [Shrinkage](https://scikit-learn.org/stable/modules/lda_qda.html#shrinkage-and-covariance-estimator) điều chuẩn ước lượng hiệp phương sai khi nhiều đặc trưng có tương quan. Điểm phân lớp có dạng:

$$
\delta_c(\mathbf z)=\mathbf z^T\Sigma_\lambda^{-1}\boldsymbol\mu_c
-\frac12\boldsymbol\mu_c^T\Sigma_\lambda^{-1}\boldsymbol\mu_c+\log\pi_c.
$$

Mô hình này là đối chứng tuyến tính để đánh giá mức phân biệt của bộ đặc trưng EEG. Learning curve thể hiện ảnh hưởng của số người huấn luyện đến balanced accuracy và F1.

In [ ]:
run_step('src.train_lda', '--device', 'auto', '--resume')
show_model('lda', ART)

### 5.2. SVM với nhân RBF

[SVM RBF](https://scikit-learn.org/stable/modules/svm.html#svm-classification) cho phép ranh giới phân loại phi tuyến trong không gian đặc trưng đã chuẩn hóa:

$$
K(\mathbf z_i,\mathbf z_j)=\exp\bigl(-\gamma\|\mathbf z_i-\mathbf z_j\|^2\bigr).
$$

Kết quả được đối chiếu với LDA và Logistic Regression để xác định liệu ranh giới phi tuyến có mang lại F1/ROC-AUC cao hơn trên cùng người kiểm tra. ROC-AUC và AP của SVM sử dụng decision score; điểm này chưa được hiệu chuẩn thành xác suất.

In [ ]:
run_step('src.train_svm', '--device', 'auto', '--resume')
show_model('svm', ART)

### 5.3. Random Forest

[Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#forest) tổng hợp xác suất dự đoán từ nhiều cây được xây dựng với lấy mẫu ngẫu nhiên và các tập đặc trưng con. Mô hình được dùng để khảo sát tương tác phi tuyến giữa các thống kê tín hiệu, đồng thời cung cấp MDI feature importance trên tập train.

Log loss theo số cây được tính trên cùng một inner split để khảo sát thay đổi dự báo khi tăng kích thước rừng. Đây không phải chuỗi objective theo vòng gradient descent; việc thêm cây không đảm bảo validation log loss giảm đơn điệu.

In [ ]:
run_step('src.train_random_forest', '--device', 'auto', '--resume')
show_model('random_forest', ART)

### 5.4. Logistic Regression với điều chuẩn L2

[Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) ước lượng $p_i=P(y_i=1\mid\mathbf z_i)=\sigma(\boldsymbol\beta^T\mathbf z_i+b)$. Hàm mục tiêu gồm logistic loss có trọng số lớp và điều chuẩn L2:

$$
\mathcal L(\boldsymbol\beta,b)=
-\sum_i w_{y_i}\bigl[y_i\log p_i+(1-y_i)\log(1-p_i)\bigr]
+\frac{\|\boldsymbol\beta\|_2^2}{2C}.
$$

So sánh với LDA giúp đánh giá hai cách xây dựng bộ phân loại tuyến tính trên cùng bộ đặc trưng. Biểu đồ tối ưu được lấy từ một lượt chẩn đoán L-BFGS riêng trên CPU, trong đó objective được chia cho $\sum_i w_{y_i}$. Log loss train/validation trên hình được tính không trọng số. Lịch sử này không đại diện cho các lượt fit GPU hoặc fit outer CV.

In [ ]:
run_step('src.train_logistic_regression', '--device', 'auto', '--resume')
show_model('logistic_regression', ART)

## 6. So sánh và thảo luận

Bảng so sánh sử dụng kết quả từ bốn lần đánh giá có cùng dữ liệu, bộ đặc trưng và cách chia người. F1 target là tiêu chí xếp hạng chính. Recall và số target bị bỏ sót cho biết mức nhạy của mô hình; precision và số standard bị nhầm thành target cho biết chi phí của việc tăng mức nhạy. ROC-AUC/AP bổ sung thông tin về khả năng xếp hạng mà không phụ thuộc riêng vào ngưỡng dự đoán đang dùng.

In [ ]:
run_step('src.compare_ml')
show_comparison(ART)

### 6.2. Giới hạn của phân tích

Các epoch trong cùng người có tương quan; vì vậy kết quả được diễn giải theo thiết kế kiểm tra người mới. ICA được ước lượng riêng từ bản ghi không nhãn của từng người, còn bộ lọc zero-phase sử dụng cả mẫu ở thời điểm tương lai. Phạm vi áp dụng là phân tích offline khi đã có bản ghi, chưa phản ánh hiệu năng triển khai trực tuyến.

Chất lượng tín hiệu được kiểm tra bằng trace, PSD, ICA và ERP. Khi dùng Fp1/Fp2 làm tín hiệu thay thế EOG, một thành phần tương quan với vùng trán có thể chứa cả hoạt động thần kinh. Các cờ kênh bất thường chỉ hỗ trợ kiểm tra; thuật toán chưa tự động phát hiện mọi dạng kênh hỏng. Những giới hạn này cần được xét khi đối chiếu P300 giữa các giai đoạn.

Cửa sổ đặc trưng một giây cho độ phân giải phổ thấp ở dải delta. Peak và latency P300 trên từng trial nhạy với nhiễu, trong khi nhiều đặc trưng phổ/wavelet có thể tương quan. MDI importance mô tả cách Random Forest sử dụng đặc trưng; bản đồ điện thế da đầu và thứ hạng đặc trưng không xác định nguồn vỏ não hoặc quan hệ nhân quả.

Thứ hạng mô hình và chênh lệch điểm trong Mục 6.1 là kết quả mô tả của cohort được phân tích. Kết luận về khác biệt giữa các thuật toán cần đánh giá xác nhận theo người hoặc một thiết kế suy luận thống kê phù hợp. Nếu tiếp tục điều chỉnh ROI, đặc trưng hay ngưỡng dự đoán từ các kết quả này, bước lựa chọn cần nằm trong inner CV và được kiểm tra trên dữ liệu độc lập. Thời gian fit chỉ có ý nghĩa trong môi trường/backend đã ghi nhận.

## Tài liệu tham khảo

1. Dzianok, P., Antonova, I., Wojciechowski, J., Dreszer, J., & Kublik, E. (2022). *The Nencki-Symfonia electroencephalography/event-related potential dataset: Multiple cognitive tasks and resting-state data collected in a sample of healthy adults*. GigaScience, 11, giac015. [DOI](https://doi.org/10.1093/gigascience/giac015).
2. MNE-Python. *Repairing artifacts with ICA*. [Tài liệu kỹ thuật](https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html).
3. scikit-learn. *Cross-validation: evaluating estimator performance — Group K-Fold*. [Tài liệu kỹ thuật](https://scikit-learn.org/stable/modules/cross_validation.html#group-k-fold).
4. scikit-learn. *Linear and Quadratic Discriminant Analysis — Shrinkage and Covariance Estimator*. [Tài liệu kỹ thuật](https://scikit-learn.org/stable/modules/lda_qda.html#shrinkage-and-covariance-estimator).
5. scikit-learn. *Support Vector Machines*. [Tài liệu kỹ thuật](https://scikit-learn.org/stable/modules/svm.html#svm-classification).
6. scikit-learn. *Ensembles — Forests of Randomized Trees*. [Tài liệu kỹ thuật](https://scikit-learn.org/stable/modules/ensemble.html#forest).
7. scikit-learn. *Linear Models — Logistic Regression*. [Tài liệu kỹ thuật](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression).